# Gaussian Continuation Data Preparation
This notebook prepares datasets for Gaussian continuation experiments using the Ackley function. Steps:
1. Generate $(X, Y_{raw})$ using the Ackley function.
2. Compute the pairwise distance matrix for $X$.
3. Select a sequence of smoothing sigmas.
4. For each sigma, compute the Gaussian kernel, normalize, and generate smoothed targets.
5. Store each $(X, Y_{\sigma_k})$ for experimentation.

In [ ]:
import numpy as np
from ma_thesis.data import ackley

# Parameters
dim = 2
N = 2000  # number of samples
x_range = (-5, 5)

# Generate random samples in 2D
def sample_ackley(n_samples, x_range):
    X = np.random.uniform(x_range[0], x_range[1], size=(n_samples, dim))
    Y_raw = ackley(X)
    return X, Y_raw

X, Y_raw = sample_ackley(N, x_range)
print(f"X shape: {X.shape}, Y_raw shape: {Y_raw.shape}")

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Compute pairwise distance matrix using k-nearest neighbors (for efficiency)
neigh = NearestNeighbors(n_neighbors=N, algorithm='auto')
neigh.fit(X)
D_dist = neigh.kneighbors_graph(X, mode='distance').toarray()

print(f"Pairwise distance matrix shape: {D_dist.shape}")

In [ ]:
# Compute mean nearest neighbor distance for sigma_max
nearest_distances, _ = neigh.kneighbors(X, n_neighbors=2)
mean_nn_dist = np.mean(nearest_distances[:, 1])

# Define sigmas: larger sigma_max for more smoothing
K = 6  # number of smoothing stages
sigma_max = 5 * mean_nn_dist  # Increase smoothing
sigmas = np.linspace(sigma_max, 0, K)
print(f"Sigmas: {sigmas}")

## Compute gaussian kernel and smoothed targets

In [ ]:
datasets = []
for sigma in sigmas:
    if sigma > 0:
        W = np.exp(-D_dist**2 / (2 * sigma**2))
    else:
        W = np.eye(N)
    W_norm = W / W.sum(axis=1, keepdims=True)
    Y_sigma = W_norm @ Y_raw
    datasets.append((X.copy(), Y_sigma.copy()))
    print(f"Sigma={sigma:.4f}: Y_sigma mean={Y_sigma.mean():.4f}, std={Y_sigma.std():.4f}")

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata

n = len(sigmas)
fig = plt.figure(figsize=(12, 4 * n))
for i in range(n):
    X_plot, Y_plot = datasets[i]
    # Surface plot
    ax1 = fig.add_subplot(n, 2, 2*i+1, projection='3d')
    xg = np.linspace(x_range[0], x_range[1], 80)
    yg = np.linspace(x_range[0], x_range[1], 80)
    Xg, Yg = np.meshgrid(xg, yg)
    Zg = griddata(X_plot, Y_plot, (Xg, Yg), method='cubic', fill_value=np.nan)
    surf = ax1.plot_surface(Xg, Yg, Zg, cmap='viridis', edgecolor='none', alpha=0.95)
    ax1.set_title(f"Sigma = {sigmas[i]:.3f}\nSurface plot")
    ax1.set_xlabel("x1")
    ax1.set_ylabel("x2")
    ax1.set_zlabel("f(x)")
    # Scatter plot
    ax2 = fig.add_subplot(n, 2, 2*i+2, projection='3d')
    ax2.scatter(X_plot[:, 0], X_plot[:, 1], Y_plot, c=Y_plot, cmap='viridis', s=10)
    ax2.set_title(f"Sigma = {sigmas[i]:.3f}\nScatter plot")
    ax2.set_xlabel("x1")
    ax2.set_ylabel("x2")
    ax2.set_zlabel("f(x)")
plt.tight_layout()
plt.show()